# Genome-Scale Metabolic Modeling -- Flux Balance Analysis (E. coli iML1515)

Simulate *E. coli*'s real, curated genome-scale metabolic network to predict growth rate and single-gene
essentiality -- a mechanistic, constraint-based model (linear optimization over real stoichiometry), not a
statistical/ML model -- then validate predicted essential genes against real experimental data from the
Keio knockout collection (via the PEC database).

~80-90% of the code below is left blank (`____`) for you to write from each cell's hint + the COBRApy docs.
Read `PROJECT_NARRATIVE.md` and `WORKFLOW.md` first (private, not in this repo's public version).

CPU-only -- no GPU needed. Every LP solve here is fast (well under a second), so this whole notebook should
run in a few minutes.

## Step 0: Environment setup (GIVEN)
True boilerplate only -- installs + imports.

In [1]:
!pip install -q cobra requests

import os
import requests
import pandas as pd
import cobra
from cobra.flux_analysis import single_gene_deletion, flux_variability_analysis

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("cobrapy version:", cobra.__version__)

cobrapy version: 0.32.1


## Step 1: Load the real iML1515 genome-scale model and describe it
**WHAT:** fetch the real, published, curated genome-scale reconstruction of *E. coli* K-12 MG1655 -- 1515
real genes, ~2712 real reactions, ~1877 real metabolites.
**WHY:** this is the real mechanistic substrate every later step operates on -- inspect it before touching
it, same discipline as every other project in this portfolio.
**HOW:** `cobra.io.load_model("iML1515")` -- COBRApy's own built-in loader, downloads the real SBML model
directly from BiGG (cached locally after the first fetch).

In [4]:
import gzip
import requests

# WORKAROUND: cobra.io.load_model()'s bundled BiGG downloader hits a 301 http->https redirect it doesn't
# follow (a real bug in that specific loader), so fetch the real SBML file directly instead.
BIGG_URL = "https://bigg.ucsd.edu/static/models/iML1515.xml.gz"
resp = requests.get(BIGG_URL)
resp.raise_for_status()

with open("iML1515.xml", "wb") as f:
    f.write(gzip.decompress(resp.content))

model = cobra.io.read_sbml_model("iML1515.xml")

# WRITE-IT: print the real number of genes, reactions, and metabolites
# hint: print(len(model.genes), len(model.reactions), len(model.metabolites))
print(len(model.genes), len(model.reactions), len(model.metabolites))

# WRITE-IT: print the real default objective (should be the biomass reaction)
# hint: print(model.objective)
print(model.objective)

# WRITE-IT: print the real default medium (exchange reaction bounds -- the nutrient environment)
# hint: print(model.medium)
print(model.medium)

1516 2712 1877
Maximize
1.0*BIOMASS_Ec_iML1515_core_75p37M - 1.0*BIOMASS_Ec_iML1515_core_75p37M_reverse_35685
{'EX_pi_e': 1000.0, 'EX_co2_e': 1000.0, 'EX_fe3_e': 1000.0, 'EX_h_e': 1000.0, 'EX_mn2_e': 1000.0, 'EX_fe2_e': 1000.0, 'EX_glc__D_e': 10.0, 'EX_zn2_e': 1000.0, 'EX_mg2_e': 1000.0, 'EX_ca2_e': 1000.0, 'EX_ni2_e': 1000.0, 'EX_cu2_e': 1000.0, 'EX_sel_e': 1000.0, 'EX_cobalt2_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_mobd_e': 1000.0, 'EX_so4_e': 1000.0, 'EX_nh4_e': 1000.0, 'EX_k_e': 1000.0, 'EX_na1_e': 1000.0, 'EX_cl_e': 1000.0, 'EX_o2_e': 1000.0, 'EX_tungs_e': 1000.0, 'EX_slnt_e': 1000.0}


## Step 2: Baseline FBA -- predict real growth rate under the default (aerobic, glucose minimal) medium
**WHAT:** solve the real linear program: maximize biomass flux subject to real stoichiometric balance and
real reaction bounds.
**HOW:** `model.optimize()` returns a `Solution` object; `.objective_value` is the real predicted growth
rate (1/hour); `.fluxes` is a real pandas Series of every reaction's real flux in this optimal solution.

In [5]:
# WRITE-IT: run real FBA and print the real predicted growth rate
# hint: solution = model.optimize()
#       print("real predicted growth rate (1/hr):", solution.objective_value)
solution = model.optimize()
print("real predicted growth rate (1/hr):", solution.objective_value)

# WRITE-IT: print the real top 10 reactions by absolute flux magnitude (what's actually "running" at
# this optimum)
# hint: print(solution.fluxes.abs().sort_values(ascending=False).head(10))
print(solution.fluxes.abs().sort_values(ascending=False).head(10))

real predicted growth rate (1/hr): 0.87699721425716
ATPS4rpp      70.432494
H2Otpp        47.162365
H2Otex        47.162365
EX_h2o_e      47.162365
CYTBO3_4pp    44.256288
NADH16pp      37.997044
CO2tpp        24.003293
CO2tex        24.003293
EX_co2_e      24.003293
O2tex         22.131763
Name: fluxes, dtype: float64


## Step 3: Real single-gene knockout screen -- predict essential genes
**WHAT:** for each of the real 1515 genes, constrain its associated reaction(s) to zero flux and re-solve
FBA -- if growth collapses (below a real threshold relative to wild-type), the gene is predicted essential.
**HOW:** `cobra.flux_analysis.single_gene_deletion(model)` runs this real screen across every gene in one
call (returns a real DataFrame of growth rate per knockout) -- don't loop gene-by-gene yourself (see
WORKFLOW.md Section 6 for why).

In [6]:
WT_GROWTH = solution.objective_value
ESSENTIALITY_THRESHOLD = 0.01  # a knockout growing below 1% of real wild-type growth is called essential

# WRITE-IT: run the real single-gene knockout screen across every gene in the model
# hint: deletion_results = single_gene_deletion(model)
deletion_results = single_gene_deletion(model)

print(deletion_results.head())

# WRITE-IT: classify each real knockout as essential/non-essential using ESSENTIALITY_THRESHOLD * WT_GROWTH
# hint: deletion_results["essential"] = deletion_results["growth"] < (ESSENTIALITY_THRESHOLD * WT_GROWTH)
deletion_results["essential"] = deletion_results["growth"] < (ESSENTIALITY_THRESHOLD * WT_GROWTH)

print("real predicted essential gene count:", deletion_results["essential"].sum())

       ids        growth   status
0  {b2557}  8.757217e-13  optimal
1  {b0118}  8.769972e-01  optimal
2  {b1920}  8.769972e-01  optimal
3  {b2066}  8.769972e-01  optimal
4  {b3739}  8.769972e-01  optimal
real predicted essential gene count: 196


## Step 4: Real Flux Variability Analysis (FVA) on central metabolism
**WHAT:** FBA's optimum is often non-unique -- FVA finds the real min/max flux each reaction can carry while
keeping growth at (or near) its optimum.
**HOW:** `flux_variability_analysis(model, reaction_list=[...], fraction_of_optimum=0.9)` on a handful of
real central-metabolism reaction IDs (e.g. glycolysis/TCA cycle reactions -- check real IDs via
`model.reactions.query("PFK")`-style search or by inspecting `solution.fluxes` from Step 2 for real
high-flux reaction names).

In [8]:
# WRITE-IT: pick a handful (4-6) of real central-metabolism reaction IDs from the model to run FVA on
# (e.g. real glycolysis/TCA reactions -- inspect model.reactions or Step 2's top-flux list for real IDs)
central_reaction_ids = ["PFK", "GAPD", "PYK", "CS", "ICDHyr", "SUCOAS", "ATPS4rpp"]
central_reaction_ids = [rid for rid in central_reaction_ids if rid in model.reactions]
print("valid reaction IDs found in model:", central_reaction_ids)

# WRITE-IT: run real FVA on those reactions at 90% of the real optimal growth rate
# hint: fva_results = flux_variability_analysis(model, reaction_list=central_reaction_ids, fraction_of_optimum=0.9)
fva_results = flux_variability_analysis(model, reaction_list=central_reaction_ids, fraction_of_optimum=0.9)
print(fva_results)

valid reaction IDs found in model: ['PFK', 'GAPD', 'PYK', 'CS', 'ICDHyr', 'SUCOAS', 'ATPS4rpp']
              minimum    maximum
PFK          0.000000  32.320587
GAPD         2.752978  18.101500
PYK          0.000000  35.558002
CS           0.848646  31.946399
ICDHyr       0.848646  13.943508
SUCOAS   -1000.000000  11.015317
ATPS4rpp    58.661673  92.230792


## Step 5: Simulate a real media shift -- anaerobic growth
**WHAT:** *E. coli* has real, well-documented biology under anaerobic conditions (loses oxidative
phosphorylation, relies on fermentation) -- a genuine biological prediction with a known real answer to
check this model against.
**HOW:** find the real oxygen exchange reaction (its BiGG ID is conventionally `EX_o2_e`), set its real
lower bound to 0 (blocking O2 uptake -- the *lower* bound, not the flux value itself, see WORKFLOW.md
Section 6), re-optimize, compare the real resulting growth rate to Step 2's aerobic baseline.

In [9]:
# WRITE-IT: find the real oxygen exchange reaction and save its original real lower bound
# hint: o2_exchange = model.reactions.get_by_id("EX_o2_e")
#       original_o2_lb = o2_exchange.lower_bound
o2_exchange = model.reactions.get_by_id("EX_o2_e")
original_o2_lb = o2_exchange.lower_bound

# WRITE-IT: block real O2 uptake (set lower bound to 0) and re-run real FBA
# hint: o2_exchange.lower_bound = 0
#       anaerobic_solution = model.optimize()
#       print("real anaerobic growth rate (1/hr):", anaerobic_solution.objective_value)
o2_exchange.lower_bound = 0 
anaerobic_solution = model.optimize()
print("real anaerobic growth rate (1/hr):", anaerobic_solution.objective_value)

# restore the real original bound so later steps aren't affected
o2_exchange.lower_bound = original_o2_lb

real anaerobic growth rate (1/hr): 0.15753981693760294


## Step 6: Fetch real experimental gene-essentiality data (PEC / Keio) and inspect it live
**WHAT:** fetch the real PEC database's essential/non-essential gene classifications, grounded largely in
the real Keio collection's physical knockout screen.
**WHY:** this is the real ground truth Step 3's predictions get validated against.
**HOW:** `requests.get("https://shigen.nig.ac.jp/ecoli/pec/download/files/PECData.dat")` -- a real,
tab-separated flat file. **Important:** the exact real column layout was not verified in advance (a sandbox
network restriction blocked checking it directly) -- print the raw real columns/first rows yourself before
assuming any column name, same discipline as #18's BV-BRC lesson (measure before patching).

In [10]:
PEC_URL = "https://shigen.nig.ac.jp/ecoli/pec/download/files/PECData.dat"

# WRITE-IT: fetch the real PEC data file
# hint: resp = requests.get(PEC_URL)
#       resp.raise_for_status()
resp = requests.get(PEC_URL)
resp.raise_for_status()

# WRITE-IT: parse it as a real tab-separated table with pandas (try header=None first and print .head() to
# see the real raw structure before assuming column names/count)
# hint: import io
#       pec_df = pd.read_csv(io.StringIO(resp.text), sep="\t", header=None)
import io
pec_df = pd.read_csv(io.StringIO(resp.text), sep="\t", header=None)
print(pec_df.head(20))
print("real shape:", pec_df.shape)

        0                                                  1      2   \
0   Orf ID  Feature Type(1:gene 2:rRNA 3:tRNA 4:ncRNA 7:tm...    Orf   
1        1                                                  1   thrL   
2        2                                                  1   thrA   
3        3                                                  1   thrB   
4        4                                                  1   thrC   
5        5                                                  1   yaaX   
6        6                                                  1   yaaA   
7        7                                                  1   yaaJ   
8        8                                                  1   talB   
9        9                                                  1    mog   
10      10                                                  1   yaaH   
11      11                                                  1   yaaW   
12      13                                                  1   

In [11]:
print(pec_df.iloc[0].tolist())   # full header row, all 13 columns
print(pec_df.iloc[1].tolist())   # one real sample data row
print(pec_df.iloc[2].tolist())

['Orf ID', 'Feature Type(1:gene 2:rRNA 3:tRNA 4:ncRNA 7:tmRNA 8:sRNA)', 'Orf', 'Alternative name', 'MapPosition', 'Start(BP)', 'End(BP)', 'Length(BP)', 'Direction(0:+ 1:-)', 'Class(1:essential 2:noessential 3:unknown)', 'PID', 'Product', 'PMID']
['1', '1', 'thrL', 'b0001,ECK0001,JW4367', '0.004095110110938549', '190', '255', '66', '0', '2', '1786182', 'thr operon leader peptide', nan]
['2', '1', 'thrA', 'b0002,ECK0002,Hs,JW0001,thrA1,thrA2,thrD', '0.0072634401731193066', '337', '2799', '2463', '0', '2', '1786183', 'fused aspartokinase I and homoserine dehydrogenase I', nan]


## Step 7: Join real PEC essentiality calls onto real iML1515 predictions and validate
**WHAT:** match PEC's real gene identifiers to iML1515's real gene IDs (both should key off real *E. coli*
b-numbers -- verify the real overlap directly, don't assume), then build a real confusion matrix comparing
predicted vs. real experimental essentiality.
**HOW:** after inspecting Step 6's real output, identify which PEC column holds the real b-number/gene ID
and which holds the real essential/non-essential classification; `set(...) & set(...)` the two real gene-ID
sets to check real overlap before joining; `sklearn.metrics.confusion_matrix` (or a real pandas crosstab) on
the joined real labels.

In [14]:
from sklearn.metrics import confusion_matrix, accuracy_score

In [15]:
# WRITE-IT: identify the real b-number/gene-ID column and the real essentiality-classification column in
# pec_df from what you printed in Step 6, build a clean real DataFrame with columns ["gene_id", "pec_essential"]
pec_data = pec_df.iloc[1:].copy()
pec_data["gene_id"] = pec_data[3].str.split(",").str[0] 
pec_data = pec_data[pec_data[9].isin(["1", "2"])]
pec_data["pec_essential"] = pec_data[9] == "1"

pec_clean = pec_data[["gene_id", "pec_essential"]].reset_index(drop=True)
print(pec_clean.head())
print("real shape after cleaning:", pec_clean.shape)

# WRITE-IT: check the real overlap between PEC's real gene IDs and iML1515's real gene IDs before joining
# hint: iml1515_gene_ids = set(g.id for g in model.genes)
#       pec_gene_ids = set(pec_clean["gene_id"])
#       print("real overlap:", len(iml1515_gene_ids & pec_gene_ids))
iml1515_gene_ids = set(g.id for g in model.genes)
pec_gene_ids = set(pec_clean["gene_id"])
print("real overlap:", len(iml1515_gene_ids & pec_gene_ids))

# WRITE-IT: join deletion_results (indexed by gene ID) with pec_clean on the real overlapping gene IDs, then
# build a real confusion matrix (predicted "essential" column vs. real "pec_essential" column)
# hint: from sklearn.metrics import confusion_matrix, accuracy_score
#       merged = deletion_results.merge(pec_clean, left_index=True, right_on="gene_id")
#       cm = confusion_matrix(merged["pec_essential"], merged["essential"])
#       acc = accuracy_score(merged["pec_essential"], merged["essential"])
#       print(cm)
#       print("real accuracy:", acc)
deletion_results["gene_id"] = deletion_results["ids"].apply(lambda s: list(s)[0])

merged = deletion_results.merge(pec_clean, on="gene_id")
cm = confusion_matrix(merged["pec_essential"], merged["essential"])
acc = accuracy_score(merged["pec_essential"], merged["essential"])
print(cm)
print("real accuracy:", acc)

  gene_id  pec_essential
0   b0001          False
1   b0002          False
2   b0003          False
3   b0004          False
4   b0005          False
real shape after cleaning: (4492, 2)
real overlap: 1441
[[1236   90]
 [  24   91]]
real accuracy: 0.9208882720333103


## Step 8: Export real results as CSV

In [16]:
# WRITE-IT: export real gene-essentiality predictions, real FVA ranges, and the real validation merge table
# hint: deletion_results.to_csv(os.path.join(RESULTS_DIR, "gene_essentiality.csv"))
#       fva_results.to_csv(os.path.join(RESULTS_DIR, "flux_variability.csv"))
#       merged.to_csv(os.path.join(RESULTS_DIR, "essentiality_validation.csv"), index=False)
deletion_results.to_csv(os.path.join(RESULTS_DIR, "gene_essentiality.csv"))
fva_results.to_csv(os.path.join(RESULTS_DIR, "flux_variability.csv"))
merged.to_csv(os.path.join(RESULTS_DIR, "essentiality_validation.csv"), index=False)

print("real results exported to", RESULTS_DIR)

real results exported to results
